In [ ]:
# import os

In [ ]:
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "YOUR_LANGSMITH_API_KEY"

In [ ]:
# # Chat ollama
# from typing import List

# from langchain.messages import AIMessage
# from langchain.tools import tool
# from langchain_ollama import ChatOllama


# @tool
# def validate_user(user_id: int, addresses: List[str]) -> bool:
#     """Validate user using historical addresses.

#     Args:
#         user_id (int): the user ID.
#         addresses (List[str]): Previous addresses as a list of strings.
#     """
#     return True


# llm = ChatOllama(
#     model="gpt-oss:20b",
#     validate_model_on_init=True,
#     temperature=0,
# ).bind_tools([validate_user])

# result = llm.invoke(
#     "Could you validate user 123? They previously lived at "
#     "123 Fake St in Boston MA and 234 Pretend Boulevard in "
#     "Houston TX."
# )

# if isinstance(result, AIMessage) and result.tool_calls:
#     print(result.tool_calls)

In [ ]:
# # Chat Embeddings
# # Create a vector store with a sample text
# from langchain_core.vectorstores import InMemoryVectorStore
# from langchain_ollama import OllamaEmbeddings

# embeddings = OllamaEmbeddings(
#     model="qwen3-embedding:8b",
#     dimensions=1024,
# )

# text = "LangChain is the framework for building context-aware reasoning applications"

# vectorstore = InMemoryVectorStore.from_texts(
#     [text],
#     embedding=embeddings,
# )

# # Use the vectorstore as a retriever
# retriever = vectorstore.as_retriever()

# # Retrieve the most similar text
# retrieved_documents = retriever.invoke("What is LangChain?")

# # Show the retrieved document's content
# print(retrieved_documents[0].page_content)

In [ ]:
# # Model Integration
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_ollama.llms import OllamaLLM

# template = """Question: {question}

# Answer: Let's think step by step."""

# prompt = ChatPromptTemplate.from_template(template)

# model = OllamaLLM(model="gemma4:e4b")

# chain = prompt | model

# chain.invoke({"question": "What is LangChain?"})

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_ollama.llms import OllamaLLM
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from langchain_core.output_parsers import StrOutputParser


video_id = "Gfr50f6ZBvo"
print("Video transcript started ......")
try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id=video_id)
    transcript = " ".join(
        item.text for item in transcript_list
    )
    print("Transcript API response successful ....")
except TranscriptsDisabled:
    print("Transcription error: Transcripts are disabled.")
    with open("demoTranscript.txt", "r", encoding="utf-8") as file:
        transcript = file.read()
except Exception as e:
    # print(
    #     f"Transcription API failed: "
    #     f"{type(e).__name__}: {e}"
    # )
    print("Fallback to demo transcript.")
    with open("demoTranscript.txt", "r", encoding="utf-8") as file:
        transcript = file.read()
print("Video transcript finished ......")





print("Splitter started ......")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.create_documents([transcript])
print("Splitter + chunk finished ......")
print(f"Total chunks created: {len(chunks)}")







print("Embeddings started .....")
embeddings = OllamaEmbeddings(
    model="embeddinggemma:300m"
)
print("Embeddings model loaded .....")
vector_store = FAISS.from_documents(
    chunks,
    embeddings,
)
print("Embeddings finished .....")
print("FAISS vector store created .....")






print("Retriever started .....")
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    },
)
print("Retriever finished .....")






print("LLM loading .....")
model = OllamaLLM(
    model="gemma4:e4b",
    temperature=0.2,
)
print("LLM loaded .....")
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer only from the provided context.
      If the content is insufficient to answer the question,
      politely say "I don't know."

      Context:
      {context}

      Question:
      {question}
    """,
    input_variables=[
        "context",
        "question",
    ],
)





def format_docs(retrieved_docs):
    """
    Convert retrieved LangChain documents
    into a single context string.
    """
    context = "\n\n".join(
        document.page_content
        for document in retrieved_docs
    )
    return context




parallel_chain = RunnableParallel(
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
)





parser = StrOutputParser()





main_chain = (parallel_chain | prompt | model | parser)
print(main_chain.get_graph().draw_ascii())


question = "Summarize the video in 5 points"

print("Running RAG query ......")

response = main_chain.invoke(question)


print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(response)

print("=" * 60)

Video transcript started ......
Fallback to demo transcript.
Video transcript finished ......
Splitter started ......
Splitter + chunk finished ......
Total chunks created: 169
Embeddings started .....
Embeddings model loaded .....
Embeddings finished .....
FAISS vector store created .....
Retriever started .....
Retriever finished .....
LLM loading .....
LLM loaded .....
            +---------------------------------+         
            | Parallel<context,question>Input |         
            +---------------------------------+         
                    **               ***                
                 ***                    **              
               **                         ***           
+----------------------+                     **         
| VectorStoreRetriever |                      *         
+----------------------+                      *         
            *                                 *         
            *                                 *        